# 00 - Project Configuration
Verify all paths, dates, thresholds, and feature list.
All settings live in `src/config.py`. Change there, reload here.

In [1]:
import warnings, os
warnings.filterwarnings('ignore')
os.environ['PYTHONWARNINGS'] = 'ignore'
warnings.filterwarnings('ignore', message='.*does not have valid feature names.*')
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
import sys
from pathlib import Path
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
import importlib, src.config as cfg
print("Project root  :", PROJECT_ROOT)
print(f"Feature count : {len(cfg.FEATURE_COLS)}")
print(f"Train cutoff  : {cfg.TRAIN_CUTOFF_DATE}")
print(f"Test cutoff   : {cfg.TEST_CUTOFF_DATE}")

Project root  : E:\Trinity College Dublin\MSc_Business_Analytics\DISSERTATION\ISD_V17
Feature count : 84
Train cutoff  : 2022-12-31
Test cutoff   : 2023-12-31


In [2]:
print("DIRECTORY STRUCTURE")
print("=" * 55)
for label, path in [("data/raw", cfg.RAW_DIR), ("data/processed", cfg.PROCESSED_DIR),
                    ("models", cfg.MODELS_DIR), ("outputs", cfg.OUTPUTS_DIR)]:
    path.mkdir(parents=True, exist_ok=True)
    print(f"  {label:<20} {path}")

DIRECTORY STRUCTURE
  data/raw             E:\Trinity College Dublin\MSc_Business_Analytics\DISSERTATION\ISD_V17\data\raw
  data/processed       E:\Trinity College Dublin\MSc_Business_Analytics\DISSERTATION\ISD_V17\data\processed
  models               E:\Trinity College Dublin\MSc_Business_Analytics\DISSERTATION\ISD_V17\models
  outputs              E:\Trinity College Dublin\MSc_Business_Analytics\DISSERTATION\ISD_V17\outputs


In [3]:
print("DATA FILE CHECK")
print("=" * 65)
all_ok = True
for name, path in cfg.RAW_FILES.items():
    if str(path).endswith('/') or name == 'nexis_dir': continue
    if Path(path).exists():
        size_mb = Path(path).stat().st_size / (1024 * 1024)
        print(f"  FOUND   | {name:<30} | {size_mb:>8.1f} MB")
    else:
        print(f"  MISSING | {name:<30} | {path}")
        all_ok = False
print()
check_special = ["cro_submissions_summary"]
for name in check_special:
    path = cfg.RAW_FILES[name]
    if Path(path).exists():
        size_mb = Path(path).stat().st_size / (1024 * 1024)
        print(f"  PRIORITY | {name:<30} | {size_mb:>8.1f} MB  (808K companies)")
    else:
        print(f"  MISSING | {name} | {cfg.RAW_FILES[name]}")
        print("           Run: python src/01_collect_cro_submissions_all.py --all-companies")
if all_ok:
    print("All data files present - ready to run.")
else:
    print("WARNING: Copy missing files into data/raw/ before continuing.")

DATA FILE CHECK
  FOUND   | company_records                |    183.5 MB
  FOUND   | fs_2022                        |     21.5 MB
  FOUND   | fs_2023                        |     12.2 MB
  FOUND   | fs_2024                        |      3.0 MB
  FOUND   | dissolutions                   |      0.6 MB
  FOUND   | cro_submissions_summary        |    116.6 MB
  FOUND   | bra11                          |      4.7 MB
  FOUND   | bra12                          |      7.8 MB
  FOUND   | bra13                          |      4.1 MB
  FOUND   | bra14                          |      7.6 MB
  FOUND   | bra15                          |     17.0 MB
  FOUND   | bra18                          |      1.9 MB
  FOUND   | bra19                          |      5.8 MB
  FOUND   | bra20                          |     18.0 MB
  FOUND   | bra30                          |      0.2 MB
  FOUND   | bra31                          |      2.7 MB
  FOUND   | bra32                          |      2.1 MB
  FOUND   | bra

In [4]:
print("TEMPORAL SPLIT DATES")
print("=" * 55)
print(f"  Training cutoff  : {cfg.TRAIN_CUTOFF_DATE}  (obs_date <= this)")
print(f"  Test cutoff      : {cfg.TEST_CUTOFF_DATE}  (obs_date <= this)")
print(f"  Observation date : {cfg.OBS_DATE_STR}  (fixed anchor)")
print(f"  Proxy cutoff     : {cfg.PROXY_CUTOFF_DATE}")

TEMPORAL SPLIT DATES
  Training cutoff  : 2022-12-31  (obs_date <= this)
  Test cutoff      : 2023-12-31  (obs_date <= this)
  Observation date : 2024-12-31  (fixed anchor)
  Proxy cutoff     : 2022-12-31


In [5]:
from src.config import FEATURE_COLS
assert len(FEATURE_COLS) == len(set(FEATURE_COLS)), "Duplicate features in FEATURE_COLS"
assert len(FEATURE_COLS) == 84, f"Expected 84 features, got {len(FEATURE_COLS)}"
print(f"Slice check PASSED -- {len(FEATURE_COLS)} features, no duplicates")

# Verify group membership
groups = {
    "CRO Filing Behaviour":   ["n_filings_yr0","filing_consistency","avg_days_to_file","max_days_to_file","sector_relative_deviation","late_filer_flag","short_period_flag","annual_submission_rate"],
    "Company Profile":        ["company_age_years","nace_enc","sector_imputed","county_enc","company_type_enc"],
    "Name Signals":           ["name_risk_score","name_generic_token_count","name_has_numbers"],
    "FAME/Director":          ["fame_days_since_accounts","fame_covered","director_dissolution_count","director_max_dissolutions","director_back_to_back_flag","director_avg_portfolio_size"],
    "Sector/County":          ["sector_death_birth_ratio","county_enterprise_density","age_vs_sector_median","sector_late_filer_risk"],
    "Network/Proxy":          ["name_address_dissolution_count","name_token_dissolution_count","name_token_dissolution_rate","same_address_dissolution_count","same_address_risk_flag","same_day_reg_count","same_day_dissolution_count"],
    "Orbis Ownership":        ["is_corporate_owned","is_foreign_owned","guo_worldcompliance","guo_irish_company_count","n_subsidiaries_ult"],
    "Orbis Financial":        ["solvency_ratio","current_ratio","is_loss_making","is_insolvent","pl_declining","sol_declining","illiquid","solvency_trend_3yr","roaa","consecutive_loss_years","ebit_margin"],
    "Orbis Operational":      ["revenue_declining","revenue_declining_2yr","is_operating_loss","ebit_declining","highly_geared","has_long_term_debt","slow_creditor_payment","employees_declining","revenue_cagr_3yr"],
    "CSO BRA Extended":       ["sector_employer_share","county_enterprise_trend","sector_birth_acceleration","sector_newborn_micro_share","sector_avg_startup_turnover"],
    "Nexis":                  ["has_negative_news_mention"],
    "CRO Charges":            ["charge_count","outstanding_charge_count","satisfied_charge_count","total_charge_events","days_since_last_charge"],
    "CRO Submissions Core":   ["director_change_count","ar_late_count","name_change_count"],
    "CRO Extended":           ["has_f8_before_obs","has_examinership_before_obs","has_winding_up_before_obs","director_resignation_count","ar_filed_count","days_since_last_ar_filing","total_submissions","submission_history_years","office_change_count","days_since_last_office_change","days_since_last_name_change","other_form_count"],
}
group_total = sum(len(v) for v in groups.values())
assert group_total == 84, f"Group total {group_total} != 84"
for gname, gfeats in groups.items():
    for f in gfeats:
        assert f in FEATURE_COLS, f"Group feature missing from FEATURE_COLS: {f}"
print(f"Group verification PASSED -- {len(groups)} groups, {group_total} features total")
print()
# File coverage check
import os
from src.config import RAW_FILES
missing = [k for k,v in RAW_FILES.items() if k != 'nexis_dir' and not str(v).endswith('.ipynb') and not os.path.exists(v)]
if missing:
    print(f"WARNING: {len(missing)} raw files not found: {missing[:3]}")
else:
    print(f"File coverage: all {len(RAW_FILES)} raw file paths defined")


Slice check PASSED -- 84 features, no duplicates
Group verification PASSED -- 14 groups, 84 features total

File coverage: all 30 raw file paths defined


In [6]:
print(f"FILING THRESHOLDS")
print("=" * 55)
print(f"  Late filer threshold : {cfg.LATE_FILER_THRESHOLD_DAYS} days")
print(f"  Statutory window     : {cfg.STATUTORY_WINDOW_DAYS} days")
print(f"  Max filing delay cap : {cfg.MAX_FILING_DELAY_DAYS} days")
print(f"  Short period         : {cfg.SHORT_PERIOD_THRESHOLD} days")
print(f"  Same-address min     : {cfg.SAME_ADDRESS_MIN_COUNT}")
print()
print(f"MODEL THINGS")
print(f"  Random state : {cfg.RANDOM_STATE}")
print(f"  Optuna       : {cfg.OPTUNA_N_TRIALS} trials")
print(f"  IF contam   : {cfg.CONTAMINATION}")
print(f"  Top N %     : {cfg.TOP_N_FLAGGED:.0%}")

FILING THRESHOLDS
  Late filer threshold : 270 days
  Statutory window     : 274 days
  Max filing delay cap : 1825 days
  Short period         : 180 days
  Same-address min     : 2

MODEL THINGS
  Random state : 42
  Optuna       : 100 trials
  IF contam   : 0.05
  Top N %     : 5%


In [ ]:
print(f"NOTEBOOK RUN ORDER")
print("=" * 65)
steps = [
    ("00_config.ipynb",                   "This file. Verify settings."),
    ("01_data_loading.ipynb",             "Pre-flight check."),
    ("02_feature_engineering_rebuilt.ipynb", f"Build all {len(cfg.FEATURE_COLS)} features."),
    ("03_model_training.ipynb",           "5 models. Equal Optuna. Calibration."),
    ("04_anomaly_detection.ipynb",        "IF + permutation test + RQ2 + RQ3."),
    ("05_shap_explainability.ipynb",      "SHAP global + per-company contributions + narratives."),
    ("06_figures.ipynb",                  "23 dissertation figures. Runs last: reads the trained model and Stage 2 outputs.")
]
for nb, desc in steps:
    print(f"  {nb}")
    print(f"  {desc}")

NOTEBOOK RUN ORDER
  00_config.ipynb
  This file. Verify settings.
  01_data_loading.ipynb
  Pre-flight check.
  02_feature_engineering_rebuilt.ipynb
  Build all 84 features.
  06_figures.ipynb
  23 dissertation figures. Runs last: reads the trained model and Stage 2 outputs.
  03_model_training.ipynb
  5 models. Equal Optuna. Calibration. ~80 min.
  04_anomaly_detection.ipynb
  IF + permutation test + RQ2 + RQ3.
  05_shap_explainability.ipynb
  SHAP global + per-company contributions + narratives.
